# Data Merging

Combine order items, orders, customers, products and sellers into `merged_order_items`. The output keeps one row per `(order_id, order_item_id)`.

**Contents**

1. [Project setup](#merge-setup) — Locate the raw inputs and interim output directory.
2. [Full item-level merge](#merge-merge) — Join the five tables and save the result.
3. [Join diagnostics](#merge-diagnostics) — Inspect join coverage, row counts and key checks.
4. [Output validation](#merge-validation) — Check the merged table against the item key, schema and reference dimensions.

Sections 3 and 4 both use the merge result from section 2 and can be inspected separately once it is in memory.

[User Guide](../docs/Merge/User_Guide.md) · [Technical Reference](../docs/Merge/Technical_Reference.md)


<a id="merge-setup" name="merge-setup"></a>

## 1. Project setup

Resolve the repository root before importing the merge API and its input and output paths.


In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import pandas as pd
from IPython.display import display


# Find the project root from the current directory
def find_project_root(start_path: Path) -> Path:
    """Find project root by locating the src/ directory and requirements.txt."""
    for path in [start_path, *start_path.parents]:
        if (path / "src").is_dir() and (path / "requirements.txt").is_file():
            return path

    raise FileNotFoundError(
        "Could not locate project root containing both 'src/' and 'requirements.txt'."
    )


PROJECT_ROOT = find_project_root(Path.cwd())


# Ensure Python recognizes the 'src' package
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))




In [ ]:
# Import project paths
from src.paths import RAW_DIR, INTERIM_DIR


# Import pipeline modules
from src.pipeline.api import merge_data
from src.pipeline.contracts.schema import BENCHMARK_ROW_COUNT, FULL_MERGE_SCHEMA
from src.pipeline.contracts.validator import validate_table_contract
from src.pipeline.merging.planner import MERGE_TABLE_SPECS
from src.pipeline.merging.reporting import build_merge_report_tables
from src.pipeline.run_merge import save_merged_dataset


# Check the environment
print(f"Project root: {PROJECT_ROOT}")
print(f"Data raw path: {RAW_DIR}")
print(f"Interim data path: {INTERIM_DIR}")

<a id="merge-merge" name="merge-merge"></a>

## 2. Full item-level merge

Start from order items. Attach order and customer attributes through the order, then add product and seller attributes using each item's keys.

### Join the source tables

Each join uses a many-to-one relationship so that an order with several items retains a separate row for each item.


In [ ]:
# Merge and validate the full registered dataset.
merge_result = merge_data(tables=list(MERGE_TABLE_SPECS))
df_merged = merge_result.data



### Save and preview the merged table

Write the item table to `data/interim` with Parquet preferred and CSV as the fallback. The preview shows the saved table's columns and first three rows.


In [ ]:
# Save to interim, trying Parquet first with CSV fallback.
saved_path = save_merged_dataset(df_merged, output_dir=INTERIM_DIR)

# Quick check of dimensions and data
print(f"Final shape: {df_merged.shape}")
df_merged.head(3)

<a id="merge-diagnostics" name="merge-diagnostics"></a>

## 3. Join diagnostics

Read the joins alongside their row counts. An unmatched item can survive a left join with missing attributes, while extra output rows would indicate that a join multiplied items.


In [ ]:
# Display output structure and checks recorded during merging.
merge_report_tables = build_merge_report_tables(merge_result)

with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.max_colwidth", None,
):
    for section, table in merge_report_tables.items():
        print(f"\n{section.replace('_', ' ').title()}")
        if table.empty:
            print("No records reported.")
        else:
            display(table)

<a id="merge-validation" name="merge-validation"></a>

## 4. Output validation

Check the item key, required values and column types on the merged table. The final comparison checks the number of item rows against both the source item table and the reference dataset.


In [ ]:
# Reference checks for the full item-level output of the current raw dataset.
base_rows = merge_result.report.get("input_shapes", {}).get("order_items", {}).get("rows")
if merge_result.schema != FULL_MERGE_SCHEMA or base_rows is None:
    raise ValueError("Run this baseline cell on the full Olist merge with its order_items report.")

# Reuse the shared column, dtype, unique-key and non-null checks.
validate_table_contract(df_merged, FULL_MERGE_SCHEMA, table_name="full_merge")

baseline_checks = pd.DataFrame(
    [
        ["Rows", BENCHMARK_ROW_COUNT, len(df_merged)],
        ["Columns", len(FULL_MERGE_SCHEMA.columns), len(df_merged.columns)],
        ["Unique orders", 98_666, df_merged["order_id"].nunique()],
        ["Order-item rows preserved", base_rows, len(df_merged)],
    ],
    columns=["Check", "Expected", "Actual"],
)
baseline_checks["Passed"] = baseline_checks["Expected"].eq(baseline_checks["Actual"])
display(baseline_checks)

if not baseline_checks["Passed"].all():
    failed_checks = ", ".join(baseline_checks.loc[~baseline_checks["Passed"], "Check"])
    raise ValueError(
        f"Full merge baseline mismatch: {failed_checks}. "
        "Check the raw dataset version and merge inputs against these reference counts."
    )

print("Full merge contract and baseline checks passed.")